In [ ]:
import sys
from pathlib import Path

repo_root = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN')
sys.path.insert(0, str(repo_root))


### 1. Configuração do Ambiente

In [ ]:
#!pip install "cognite-sdk[pandas]" matplotlib seaborn tensorflow plotly -q

import os
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display
from sklearn.preprocessing import RobustScaler
from statsmodels.tsa.stattools import acf
from industrial_ts.dataloader import DataLoader
import json
from getpass import getpass
#import tensorflow as tf
#from tensorflow import keras

print("Bibliotecas importadas com sucesso!")






In [ ]:
import debugpy
debugpy.listen(('0.0.0.0', 5678))






In [ ]:
os.environ['COGNITE_CLIENT_SECRET'] = getpass("Enter COGNITE_CLIENT")






### 2. Ativar o DataLoader

In [ ]:
import importlib
import sys
importlib.reload(sys.modules['industrial_ts.dataloader'])
from industrial_ts.dataloader import DataLoader
dl = DataLoader()
dl.add_segments(segments=3, window=10, step=10, series=[
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActShaft Power',
    'PH (CBM) 1st Stg ActCompr Poly Head',
    'PH (CBM) 1st Stage ActPress Ratio'  
], path="segmenter_model_10.pkl"
)
#dl.segmenter.save("segmenter_model_10.pkl")







### 10 Pós-processamento

In [ ]:
for i in [2]:
    dl.df['states'].replace(i, 1, inplace=True) # Merge states 1 and 2







In [ ]:
dl.add_time_to_change_state_timestamp()






## Baseline TSDF_GRU

In [ ]:
from pathlib import Path
import json
import time
from industrial_ts.gru import TSDF_GRU

# --- Resumo para salvar no JSON ---
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }

# Best epoch by test micro
def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')


# --- Nome do arquivo baseado na configuracao ---
def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_gru(cfg):
    tp = cfg.get('train_params', {})
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    lr = tp.get('optimizer_params', {}).get('lr', 'lr')
    return _safe(f"gru_{clamp}_{opt}_lr{lr}")


# --- Config base (se nao existir do TSDF_seqKAN) ---
if 'run_config' not in globals():
    batch_size = 512
    run_config = dict(
        model='TSDF_seqKAN',
        in_channels=11,
        hidden_dim=11*32,
        cost_columns=[
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActShaft Power',
            'PH (CBM) 1st Stg ActCompr Poly Head',
            'PH (CBM) 1st Stage ActPress Ratio'
        ],
        lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        sigma_temp=0.6,
        log_likelihood=False,
        use_layernorm=False,
        direct_x=True,
        train_params=dict(
            batch_size=batch_size,
            window_size=10,
            window_step=10,
            epochs=150,
            validate=True,
            patience=20,
            kl_warmup_epochs=20,
            kl_start=0.01,
            rebuild=True,
            reconstruction_test=False,
            warmup_steps=0,
            min_lr_factor=1.0,
            optimizer_name='radam',
            optimizer_params={'lr': 2e-4},
            grad_clip_max_norm=0.3,
            debug_batch_stats=True,
            x_min=-3.0,
            x_max=3.0,
        ),
    )

# --- defaults se run_config veio de outra celula ---
if 'cost_columns' not in run_config:
    run_config['cost_columns'] = [
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio'
    ]
if 'lam' not in run_config:
    run_config['lam'] = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
if 'sigma_temp' not in run_config:
    run_config['sigma_temp'] = 0.6
if 'log_likelihood' not in run_config:
    run_config['log_likelihood'] = False
if 'use_layernorm' not in run_config:
    run_config['use_layernorm'] = False
if 'train_params' not in run_config:
    run_config['train_params'] = {}

run_config_gru = dict(
    model='TSDF_GRU',
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config['cost_columns'],
    lam=run_config['lam'],
    sigma_temp=run_config['sigma_temp'],
    log_likelihood=run_config['log_likelihood'],
    use_layernorm=run_config['use_layernorm'],
    train_params=run_config['train_params'],
)

run_name_gru = _make_run_name_gru(run_config_gru)
out_json_gru = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name_gru}.json")

# Garante sem clamp para GRU

model_gru = TSDF_GRU(
    in_channels=run_config_gru['in_channels'],
    hidden_dim=run_config_gru['hidden_dim'],
    cost_columns=run_config_gru['cost_columns'],
    lam=run_config_gru['lam'],
    sigma_temp=run_config_gru['sigma_temp'],
    log_likelihood=run_config_gru['log_likelihood'],
    use_layernorm=run_config_gru['use_layernorm'],
)

train_start_gru = time.time()
res_gru = model_gru.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config_gru['train_params']['batch_size'],
    window_size=run_config_gru['train_params']['window_size'],
    window_step=run_config_gru['train_params']['window_step'],
    epochs=run_config_gru['train_params']['epochs'],
    validate=run_config_gru['train_params']['validate'],
    patience=run_config_gru['train_params']['patience'],
    kl_warmup_epochs=run_config_gru['train_params']['kl_warmup_epochs'],
    kl_start=run_config_gru['train_params']['kl_start'],
    rebuild=run_config_gru['train_params']['rebuild'],
    reconstruction_test=run_config_gru['train_params']['reconstruction_test'],
    warmup_steps=run_config_gru['train_params']['warmup_steps'],
    min_lr_factor=run_config_gru['train_params']['min_lr_factor'],
    optimizer_name=run_config_gru['train_params']['optimizer_name'],
    optimizer_params=run_config_gru['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config_gru['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config_gru['train_params']['debug_batch_stats'],
        debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config_gru['train_params']['x_min'],
    x_max=run_config_gru['train_params']['x_max'],
)
train_end_gru = time.time()
train_seconds_gru = train_end_gru - train_start_gru

res_gru = [r for r in res_gru if r is not None]
out_dir = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results')
out_dir.mkdir(parents=True, exist_ok=True)
gru_ckpt = out_dir / f"{run_name_gru}.pt"
model_gru.save(str(gru_ckpt))
print('saved model:', gru_ckpt)

# Best epoch by test micro
best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_gru)
summary = _extract_summary(res_gru)

with open(out_json_gru, 'w') as f:
    payload = {
        'config': run_config_gru,
        'results': res_gru,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds_gru,
        'best_epoch': summary['best_epoch'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    json.dump(payload, f)

print(res_gru[-1] if res_gru else 'no results')
print('saved:', out_json_gru)



# Baseline TS_GRU


In [ ]:
import importlib
import industrial_ts.seqKAN as sk
importlib.reload(sk)
from industrial_ts.seqKAN import TS_seqKANSeq


In [ ]:
# Baseline TS_GRU (mesmo pipeline do ODEJump)
from industrial_ts.gru import TS_GRU
import time
import json as _json
from pathlib import Path

run_config_gru = dict(
    model='TS_GRU',
    in_channels=11,
    hidden_dim=24,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio',
    ],
    lam=[1.0, 0.0, 0.0, 0.0],
    train_params=dict(
        batch_size=512,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        seed_split=42,
        fixed_test_idx=None,
        lr=2e-4,
        x_min=-3.0,
        x_max=3.0,
        debug_batch_stats=True,
        debug_batch_stats_names=list(dl.df.columns[:-3]),
        timestamp_col='index',
        states_col='states',
        static_features_cols=None,
    ),
)

def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_gru(cfg):
    tp = cfg.get('train_params', {})
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    h = cfg.get('hidden_dim', 'H?')
    return _safe(f"gru_H{h}_{clamp}")

run_name_gru = _make_run_name_gru(run_config_gru)
out_json_gru = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name_gru}.json")

model_gru = TS_GRU(
    in_channels=run_config_gru['in_channels'],
    hidden_dim=run_config_gru['hidden_dim'],
    cost_columns=run_config_gru.get('cost_columns'),
    lam=run_config_gru.get('lam'),
)

train_start_gru = time.time()
res_gru = model_gru.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=run_config_gru['train_params'].get('static_features_cols'),
    timestamp_col=run_config_gru['train_params'].get('timestamp_col', 'index'),
    states_col=run_config_gru['train_params'].get('states_col', 'states'),
    batch_size=run_config_gru['train_params']['batch_size'],
    window_size=run_config_gru['train_params']['window_size'],
    window_step=run_config_gru['train_params']['window_step'],
    epochs=run_config_gru['train_params']['epochs'],
    validate=run_config_gru['train_params']['validate'],
    patience=run_config_gru['train_params']['patience'],
    seed_split=run_config_gru['train_params']['seed_split'],
    fixed_test_idx=run_config_gru['train_params']['fixed_test_idx'],
    lr=run_config_gru['train_params'].get('lr', 2e-4),
    x_min=run_config_gru['train_params'].get('x_min', None),
    x_max=run_config_gru['train_params'].get('x_max', None),
        debug_batch_stats=True,
        debug_batch_stats_names=list(dl.df.columns[:-3]),
)
train_end_gru = time.time()
train_seconds_gru = train_end_gru - train_start_gru

res_gru = [r for r in res_gru if r is not None]
out_dir = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results')
out_dir.mkdir(parents=True, exist_ok=True)
gru_ckpt = out_dir / f"{run_name_gru}.pt"
model_gru.save(str(gru_ckpt))
print('saved model:', gru_ckpt)

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_gru)
summary = _extract_summary(res_gru)
print(f"train_L1 best/final: {summary['L1_best']} / {summary['L1_final']}")


# fallback de best_epoch se nao vier no summary
if summary['best_epoch'] is None:
    be, _ = _best_epoch_by_test_micro(res_gru)
    summary['best_epoch'] = be
with open(out_json_gru, 'w') as f:
    payload = {
        'config': run_config_gru,
        'results': res_gru,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds_gru,
        'best_epoch': summary['best_epoch'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    _json.dump(payload, f)

print(res_gru[-1] if res_gru else 'no results')
print('saved:', out_json_gru)

# TS_seqKANseq

In [ ]:
import importlib
import industrial_ts.seqKAN as sk
importlib.reload(sk)
from industrial_ts.seqKAN import TS_seqKANSeq


In [ ]:
# Treino via train_cognite com TS_seqKANSeq (mesmo pipeline do TS_GRU)
from industrial_ts.seqKAN import TS_seqKANSeq
import time
import json as _json
from pathlib import Path

# hiperparametros KAN
grid_eps = 0.02
kan_params_seq = {
    'cell': {'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'output': {'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
    'topk': {
        'enabled': True,
        'k_x': 4,
        'k_h': 6,
        'k_out': 4,
        'warmup_epochs': 30,
        'mode': 'soft',
        'temp': 0.5,
    },
}

batch_size = 512

run_config = dict(
    model='TS_seqKANSeq',
    in_channels=11,
    hidden_dim=24,
    lam=[1.0, 0.0, 0.0, 0.0],
    kan_params=kan_params_seq,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio',
    ],
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        seed_split=42,
        fixed_test_idx=None,
        lr=2e-4,
        x_min=-3.0,
        x_max=3.0,
        debug_batch_stats=True,
        debug_batch_stats_names=list(dl.df.columns[:-3]),
        timestamp_col='index',
        states_col='states',
        static_features_cols=None,
    ),
)

def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_seqkanseq(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_c = kp.get('cell', {}).get('grid', 'g?')
    grid_eps = kp.get('cell', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    h = cfg.get('hidden_dim', 'H?')
    return _safe(f"seqKANseq_H{h}_g{grid_c}_ge{grid_eps}_{clamp}")

run_name = _make_run_name_seqkanseq(run_config)
out_json = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name}.json")

model_seq = TS_seqKANSeq(
    in_channels=run_config['in_channels'],
    hidden_dim=run_config['hidden_dim'],
    cost_columns=run_config.get('cost_columns'),
    lam=run_config.get('lam'),
    kan_params=run_config['kan_params'],
)

train_start = time.time()
res = model_seq.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=run_config['train_params'].get('static_features_cols'),
    timestamp_col=run_config['train_params'].get('timestamp_col', 'index'),
    states_col=run_config['train_params'].get('states_col', 'states'),
    batch_size=run_config['train_params']['batch_size'],
    window_size=run_config['train_params']['window_size'],
    window_step=run_config['train_params']['window_step'],
    epochs=run_config['train_params']['epochs'],
    validate=run_config['train_params']['validate'],
    patience=run_config['train_params']['patience'],
    seed_split=run_config['train_params']['seed_split'],
    fixed_test_idx=run_config['train_params']['fixed_test_idx'],
    lr=run_config['train_params'].get('lr', 2e-4),
    x_min=run_config['train_params'].get('x_min', None),
    x_max=run_config['train_params'].get('x_max', None),
        debug_batch_stats=True,
        debug_batch_stats_names=list(dl.df.columns[:-3]),
)
train_end = time.time()
train_seconds = train_end - train_start

res = [r for r in res if r is not None]
out_dir = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results')
out_dir.mkdir(parents=True, exist_ok=True)
seqkanseq_ckpt = out_dir / f"{run_name}.pt"
import torch
torch.save(model_seq.state_dict(), seqkanseq_ckpt)
print('saved model:', seqkanseq_ckpt)

def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }

summary = _extract_summary(res)
print(f"train_L1 best/final: {summary['L1_best']} / {summary['L1_final']}")

def _json_safe(o):
    import numpy as _np
    if isinstance(o, dict):
        return {k: _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json_safe(v) for v in o]
    if isinstance(o, _np.ndarray):
        return o.tolist()
    if isinstance(o, _np.floating):
        return float(o)
    if isinstance(o, _np.integer):
        return int(o)
    return o

best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res)

with open(out_json, 'w') as f:
    payload = {
        'config': run_config,
        'results': res,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds,
        'best_epoch': summary['best_epoch'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    _json.dump(_json_safe(payload), f)

print(res[-1] if res else 'no results')
print('saved:', out_json)


# TSDF_seqKANseq

In [ ]:
# Treino via train_cognite com TSDF_seqKANSeq
from industrial_ts.seqKAN import TSDF_seqKANSeq
import time
import json as _json
from pathlib import Path

def _safe(s):
    return ''.join(c if c.isalnum() or c in '-_.' else '_' for c in str(s))

def _make_run_name_tsdf_seqkanseq(cfg):
    tp = cfg.get('train_params', {})
    kp = cfg.get('kan_params', {})
    grid_eps = kp.get('cell', {}).get('grid_eps', kp.get('grid_eps', 'ge?'))
    clamp = f"c{tp.get('x_min')}_{tp.get('x_max')}"
    opt = _safe(tp.get('optimizer_name', 'opt'))
    h = cfg.get('hidden_dim', 'H?')
    return _safe(f"tsdf_seqKANseq_H{h}_ge{grid_eps}_{clamp}_{opt}")

# config standalone (nao depende de outras celulas)
if 'kan_params_seq' not in globals():
    grid_eps = globals().get('grid_eps', 0.2)
    kan_params_seq = {
        'cell': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
        'output': { 'grid_eps': grid_eps, 'grid': 8, 'k': 3, 'grid_range': (-3, 3)},
        'topk': {
            'enabled': True,
            'k_x': 4,
            'k_h': 6,
            'k_out': 4,
            'warmup_epochs': 30,
            'mode': 'soft',
            'temp': 0.5,
        },
    }

batch_size = 512
train_fraction = 0.6

run_config_tsdf_seqkanseq = dict(
    model='TSDF_seqKANSeq',
    in_channels=11,
    hidden_dim=24,
    cost_columns=[
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActShaft Power',
        'PH (CBM) 1st Stg ActCompr Poly Head',
        'PH (CBM) 1st Stage ActPress Ratio',
    ],
    lam=[1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    sigma_temp=0.6,
    log_likelihood=False,
    use_layernorm=False,
    direct_x=True,
    kan_params=kan_params_seq,
    train_params=dict(
        batch_size=batch_size,
        window_size=10,
        window_step=10,
        epochs=150,
        validate=True,
        patience=20,
        kl_warmup_epochs=20,
        kl_start=0.01,
        rebuild=True,
        reconstruction_test=False,
        warmup_steps=0,
        min_lr_factor=1.0,
        optimizer_name='radam',
        optimizer_params={'lr': 2e-4},
        grad_clip_max_norm=0.3,
        debug_batch_stats=True,
        x_min=-3.0,
        x_max=3.0,
    ),
)

run_name_tsdf_seqkanseq = _make_run_name_tsdf_seqkanseq(run_config_tsdf_seqkanseq)
out_json_tsdf_seqkanseq = str(Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results') / f"final_rebuild_{run_name_tsdf_seqkanseq}.json")

# --- Resumo para salvar no JSON (mesmo padrao do GRU) ---
def _extract_summary(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return {
            'best_epoch': None,
            'L1_best': None,
            'L1_final': None,
            'Test_best': None,
            'Test_final': None,
            'per_group_best': None,
            'per_group_final': None,
            'best_val_epoch': None,
            'best_val_micro': None,
        }

    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    last = metrics[-1]

    if 'val_micro' in metrics[0]:
        best_val = min(metrics, key=lambda m: m.get('val_micro', float('inf')))
    else:
        best_val = None

    def _pack_test(m):
        if m is None:
            return None
        return {
            'micro_mse': m.get('micro_mse'),
            'micro_se': m.get('micro_se'),
            'macro_mse': m.get('macro_mse'),
            'macro_se': m.get('macro_se'),
        }

    def _pack_groups(m):
        if m is None:
            return None
        return {
            'mse': m.get('per_group_mse'),
            'se_w': m.get('per_group_se_w'),
            'se_unw': m.get('per_group_se_unw'),
            'counts': m.get('per_group_counts'),
            'sum_nobs': m.get('per_group_sum_nobs'),
        }

    return {
        'best_epoch': best.get('epoch'),
        'L1_best': best.get('train_L1'),
        'L1_final': last.get('train_L1'),
        'Test_best': _pack_test(best),
        'Test_final': _pack_test(last),
        'per_group_best': _pack_groups(best),
        'per_group_final': _pack_groups(last),
        'best_val_epoch': best_val.get('epoch') if best_val is not None else None,
        'best_val_micro': best_val.get('val_micro') if best_val is not None else None,
    }

def _best_epoch_by_test_micro(res):
    metrics = [r for r in res if r is not None and isinstance(r, dict)]
    if not metrics:
        return None, None
    best = min(metrics, key=lambda m: m.get('micro_mse', float('inf')))
    return best.get('epoch'), best.get('micro_mse')

model_tsdf_seqkanseq = TSDF_seqKANSeq(
    in_channels=run_config_tsdf_seqkanseq['in_channels'],
    hidden_dim=run_config_tsdf_seqkanseq['hidden_dim'],
    cost_columns=run_config_tsdf_seqkanseq['cost_columns'],
    lam=run_config_tsdf_seqkanseq['lam'],
    sigma_temp=run_config_tsdf_seqkanseq['sigma_temp'],
    log_likelihood=run_config_tsdf_seqkanseq['log_likelihood'],
    use_layernorm=run_config_tsdf_seqkanseq['use_layernorm'],
    direct_x=run_config_tsdf_seqkanseq['direct_x'],
    kan_params=run_config_tsdf_seqkanseq['kan_params'],
)

train_start_tsdf_seqkanseq = time.time()
res_tsdf_seqkanseq = model_tsdf_seqkanseq.train_cognite(
    df=dl.df,
    feature_cols=list(dl.df.columns[:-3]),
    static_features_cols=None,
    timestamp_col='index',
    states_col='states',
    batch_size=run_config_tsdf_seqkanseq['train_params']['batch_size'],
    window_size=run_config_tsdf_seqkanseq['train_params']['window_size'],
    window_step=run_config_tsdf_seqkanseq['train_params']['window_step'],
    epochs=run_config_tsdf_seqkanseq['train_params']['epochs'],
    validate=run_config_tsdf_seqkanseq['train_params']['validate'],
    patience=run_config_tsdf_seqkanseq['train_params']['patience'],
    kl_warmup_epochs=run_config_tsdf_seqkanseq['train_params']['kl_warmup_epochs'],
    kl_start=run_config_tsdf_seqkanseq['train_params']['kl_start'],
    rebuild=run_config_tsdf_seqkanseq['train_params']['rebuild'],
    reconstruction_test=run_config_tsdf_seqkanseq['train_params']['reconstruction_test'],
    warmup_steps=run_config_tsdf_seqkanseq['train_params']['warmup_steps'],
    min_lr_factor=run_config_tsdf_seqkanseq['train_params']['min_lr_factor'],
    optimizer_name=run_config_tsdf_seqkanseq['train_params']['optimizer_name'],
    optimizer_params=run_config_tsdf_seqkanseq['train_params']['optimizer_params'],
    grad_clip_max_norm=run_config_tsdf_seqkanseq['train_params']['grad_clip_max_norm'],
    debug_batch_stats=run_config_tsdf_seqkanseq['train_params']['debug_batch_stats'],
        debug_batch_stats_names=list(dl.df.columns[:-3]),
    x_min=run_config_tsdf_seqkanseq['train_params']['x_min'],
    x_max=run_config_tsdf_seqkanseq['train_params']['x_max'],
)
train_end_tsdf_seqkanseq = time.time()
train_seconds_tsdf_seqkanseq = train_end_tsdf_seqkanseq - train_start_tsdf_seqkanseq

res_tsdf_seqkanseq = [r for r in res_tsdf_seqkanseq if r is not None]

out_dir = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results')
out_dir.mkdir(parents=True, exist_ok=True)
tsdf_seqkanseq_ckpt = out_dir / f"{run_name_tsdf_seqkanseq}.pt"
import torch
torch.save(model_tsdf_seqkanseq.state_dict(), tsdf_seqkanseq_ckpt)
print('saved model:', tsdf_seqkanseq_ckpt)

# resumo no mesmo padrao do GRU
best_epoch_test_micro, best_test_micro = _best_epoch_by_test_micro(res_tsdf_seqkanseq)
summary = _extract_summary(res_tsdf_seqkanseq)

with open(out_json_tsdf_seqkanseq, 'w') as f:
    payload = {
        'config': run_config_tsdf_seqkanseq,
        'results': res_tsdf_seqkanseq,
        'best_epoch_test_micro': best_epoch_test_micro,
        'best_test_micro': best_test_micro,
        'train_seconds': train_seconds_tsdf_seqkanseq,
        'best_epoch': summary['best_epoch'],
        'best_val_epoch': summary['best_val_epoch'],
        'best_val_micro': summary['best_val_micro'],
        'L1_best': summary['L1_best'],
        'L1_final': summary['L1_final'],
        'Test_best': summary['Test_best'],
        'Test_final': summary['Test_final'],
        'per_group_best': summary['per_group_best'],
        'per_group_final': summary['per_group_final'],
    }
    _json.dump(payload, f)

print(res_tsdf_seqkanseq[-1] if res_tsdf_seqkanseq else 'no results')
print('saved:', out_json_tsdf_seqkanseq)


# Valores fora do clamp

In [ ]:
import numpy as np
import torch
from industrial_ts.ode_jump import ODEJump

X = dl.df.iloc[:, :-3].to_numpy(dtype=float)

train_idx = np.arange(int(0.6 * len(X)))

X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # (N, 1, C)
X_scaled = ODEJump.scale_tensor(X_t[train_idx], X_t).squeeze(1).cpu().numpy()

total = X_scaled.size
abs_x = np.abs(X_scaled)
count_out = int((abs_x > 3).sum())
pct_out = 100.0 * count_out / total

p1, p50, p99 = np.percentile(X_scaled, [1, 50, 99])
x_min = float(X_scaled.min())
x_max = float(X_scaled.max())

print(f"Total valores: {total}")
print(f"|x| > 3: {count_out} ({pct_out:.2f}%)")
print(f"min: {x_min:.3f} max: {x_max:.3f}")
print(f"p1/p50/p99: [{p1:.6f} {p50:.6f} {p99:.6f}]")


In [ ]:
import numpy as np
import torch
from industrial_ts.ode_jump import ODEJump

# dados e nomes
cols = list(dl.df.columns[:-3])
X = dl.df.iloc[:, :-3].to_numpy(dtype=float)

# mesma escala do modelo (fit no treino)
train_idx = np.arange(int(0.6 * len(X)))  # ajuste se tiver train_idx real
X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # (N,1,C)
X_scaled = ODEJump.scale_tensor(X_t[train_idx], X_t).squeeze(1).cpu().numpy()

# métricas por feature
absX = np.abs(X_scaled)
pct_gt3 = (absX > 3).mean(axis=0) * 100
mins = X_scaled.min(axis=0)
maxs = X_scaled.max(axis=0)
p99 = np.percentile(X_scaled, 99, axis=0)

# ordena por maior %>|3|
order = np.argsort(-pct_gt3)

print("top features por %|x|>3 (scaled):")
for i in order[:20]:
    print(f"{cols[i]:60s}  %>|3|={pct_gt3[i]:6.2f}  min={mins[i]:8.3f}  max={maxs[i]:8.3f}  p99={p99[i]:8.3f}")


In [ ]:
import numpy as np
import torch
from industrial_ts.ode_jump import ODEJump

cols = list(dl.df.columns[:-3])
feat = "PH (CBM) 1st Stage ExpPress Ratio"

X = dl.df.iloc[:, :-3].to_numpy(dtype=float)
train_idx = np.arange(int(0.6 * len(X)))  # ajuste se tiver o train_idx real
X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
X_scaled = ODEJump.scale_tensor(X_t[train_idx], X_t).squeeze(1).cpu().numpy()

j = cols.index(feat)
mask = np.abs(X_scaled[:, j]) > 3

count = int(mask.sum())
total = len(mask)
pct = 100.0 * count / total

print(f"{feat}")
print(f"Total timestamps: {total}")
print(f"|x| > 3: {count} ({pct:.2f}%)")

# mostra alguns timestamps e valores originais/escalados
idx = np.where(mask)[0][:10]
print("exemplos (index, raw, scaled):")
for i in idx:
    raw = X[i, j]
    sc = X_scaled[i, j]
    ts = dl.df.index[i]  # se o índice for datetime
    print(i, ts, f"raw={raw:.6f}", f"scaled={sc:.6f}")


# Extrair parametros das splines do seqKANseq  

 Requer que `model_seq` (TS_seqKANSeq) ja exista

In [ ]:

import json
from pathlib import Path

out_path = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results/seqKANseq_splines.json')


def _kanlayer_to_dict(kl):
    d = {}
    d['in_dim'] = int(getattr(kl, 'in_dim', -1))
    d['out_dim'] = int(getattr(kl, 'out_dim', -1))
    d['num'] = int(getattr(kl, 'num', -1))
    d['k'] = int(getattr(kl, 'k', -1))
    d['grid'] = kl.grid.detach().cpu().tolist() if hasattr(kl, 'grid') else None
    d['coef'] = kl.coef.detach().cpu().tolist() if hasattr(kl, 'coef') else None
    d['scale_sp'] = kl.scale_sp.detach().cpu().tolist() if hasattr(kl, 'scale_sp') else None
    d['scale_base'] = kl.scale_base.detach().cpu().tolist() if hasattr(kl, 'scale_base') else None
    d['mask'] = kl.mask.detach().cpu().tolist() if hasattr(kl, 'mask') else None
    d['base_fun'] = kl.base_fun.__class__.__name__ if hasattr(kl, 'base_fun') else None
    return d


def _multkan_to_dict(kan):
    # ArticleKAN (MultKAN) -> lista de KANLayer em act_fun
    out = {
        'class': kan.__class__.__name__,
        'width': getattr(kan, 'width', None),
        'grid_eps': getattr(kan, 'grid_eps', None),
        'grid_range': getattr(kan, 'grid_range', None),
        'base_fun_name': getattr(kan, 'base_fun_name', None),
        'layers': [],
    }
    act_fun = getattr(kan, 'act_fun', None)
    if act_fun is not None:
        for i, layer in enumerate(act_fun):
            # act_fun pode ter lista de KANLayer
            out['layers'].append({'layer_index': i, **_kanlayer_to_dict(layer)})
    return out


payload = {
    'kan_cell': _multkan_to_dict(model_seq.seq.kan_cell),
    'kan_out': [_multkan_to_dict(k) for k in model_seq.seq.kan_out],
}

# se existir cabeca de rebuild
if getattr(model_seq.seq, 'kan_out_rebuild', None) is not None:
    payload['kan_out_rebuild'] = [_multkan_to_dict(k) for k in model_seq.seq.kan_out_rebuild]

out_path.write_text(json.dumps(payload, indent=2))
print('saved:', out_path)


In [ ]:
# Exportar formulas (LaTeX) das splines do seqKANseq (todas as conexoes)
import json
from pathlib import Path

spl_path = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results/seqKANseq_splines.json')
out_tex = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results/seqKANseq_splines.tex')

data = json.loads(spl_path.read_text())

# Representacao simbolica simples: sum_j c_j B_{j,k}(x) + base
# (nao expande o B-spline em polinomio por intervalo)

def _latex_for_layer(layer, prefix):
    lines = []
    in_dim = layer.get('in_dim', 0)
    out_dim = layer.get('out_dim', 0)
    k = layer.get('k', None)
    grid = layer.get('grid', None)
    coef = layer.get('coef', None)
    scale_sp = layer.get('scale_sp', None)
    scale_base = layer.get('scale_base', None)
    base_fun = layer.get('base_fun', 'base')
    if coef is None:
        return lines
    for i in range(in_dim):
        for o in range(out_dim):
            c = coef[i][o]
            sp = scale_sp[i][o] if scale_sp is not None else 1.0
            sb = scale_base[i][o] if scale_base is not None else 0.0
            # LaTeX string
            expr = f"f_{{{prefix},{i}\to{o}}}(x)= {sb:.6g}\\,\\phi(x) + {sp:.6g}\\,\\sum_{{j=0}}^{{{len(c)-1}}} c_j B_{{j,{k}}}(x)"
            lines.append(expr)
    # grid info
    if grid is not None:
        lines.append("\\\\")
        lines.append(f"% grid points for {prefix}: {grid}")
    return lines

tex_lines = []
tex_lines.append("% seqKANseq spline formulas (symbolic B-spline form)")

# kan_cell
tex_lines.append("% ---- kan_cell ----")
tex_lines.extend(_latex_for_layer(data['kan_cell']['layers'][0], 'cell'))

# kan_out (cada head tem uma MultKAN com layers)
tex_lines.append("% ---- kan_out heads ----")
for h, head in enumerate(data.get('kan_out', [])):
    for li, layer in enumerate(head.get('layers', [])):
        tex_lines.append(f"% head {h}, layer {li}")
        tex_lines.extend(_latex_for_layer(layer, f"out{h}_L{li}"))

# kan_out_rebuild (se existir)
if 'kan_out_rebuild' in data:
    tex_lines.append("% ---- kan_out_rebuild heads ----")
    for h, head in enumerate(data.get('kan_out_rebuild', [])):
        for li, layer in enumerate(head.get('layers', [])):
            tex_lines.append(f"% rebuild head {h}, layer {li}")
            tex_lines.extend(_latex_for_layer(layer, f"rebuild{h}_L{li}"))

out_tex.write_text("\n".join(tex_lines))
print('saved:', out_tex)


In [ ]:
# Analise de falha no sinal pelas formulas (4 criterios)
# 1) Fora do intervalo esperado
# 2) Curvatura alta
# 3) Oscilacao alta
# 4) Instabilidade entre nos

import json
import numpy as np
import torch
from pathlib import Path

from seqkan.kan.spline import coef2curve

spl_path = Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results/seqKANseq_splines.json')
data = json.loads(spl_path.read_text())

# parametros
x_min, x_max = -3.0, 3.0
num_samples = 256

x = torch.linspace(x_min, x_max, steps=num_samples).unsqueeze(1)  # (N,1)


def _eval_spline(grid, coef, k):
    # grid: (in_dim, G+2k), coef: (in_dim,out_dim,G+k)
    grid_t = torch.tensor(grid, dtype=torch.float32)
    coef_t = torch.tensor(coef, dtype=torch.float32)
    # avaliando apenas in_dim=1 -> usa x com shape (N,1)
    y = coef2curve(x, grid_t, coef_t, k)  # (N, in_dim, out_dim)
    return y.squeeze(1)  # (N, out_dim)


def _metrics(y):
    y_np = y.detach().cpu().numpy()
    # criterio 1: fora do intervalo
    out_of_range = np.mean((y_np < x_min) | (y_np > x_max))
    # criterio 2: curvatura (2a derivada aprox)
    dy = np.gradient(y_np, axis=0)
    d2y = np.gradient(dy, axis=0)
    curvature = np.mean(np.abs(d2y))
    # criterio 3: oscilacao (variacao total)
    total_var = np.mean(np.abs(np.diff(y_np, axis=0)))
    # criterio 4: instabilidade entre nos (max salto local)
    max_jump = np.max(np.abs(np.diff(y_np, axis=0)))
    return out_of_range, curvature, total_var, max_jump


def _rank_layers(layer, prefix):
    in_dim = layer.get('in_dim', 0)
    out_dim = layer.get('out_dim', 0)
    k = layer.get('k', None)
    grid = layer.get('grid', None)
    coef = layer.get('coef', None)
    if grid is None or coef is None:
        return []
    # calcula por conexao
    rows = []
    for i in range(in_dim):
        for o in range(out_dim):
            y = _eval_spline([grid[i]], [[coef[i][o]]], k)  # y: (N,1)
            oor, curv, tv, jump = _metrics(y)
            rows.append((f"{prefix}:{i}->{o}", oor, curv, tv, jump))
    return rows

rows = []
# kan_cell
rows.extend(_rank_layers(data['kan_cell']['layers'][0], 'cell'))
# kan_out
for h, head in enumerate(data.get('kan_out', [])):
    for li, layer in enumerate(head.get('layers', [])):
        rows.extend(_rank_layers(layer, f"out{h}_L{li}"))
# kan_out_rebuild
for h, head in enumerate(data.get('kan_out_rebuild', [])):
    for li, layer in enumerate(head.get('layers', [])):
        rows.extend(_rank_layers(layer, f"rebuild{h}_L{li}"))

# ordena por cada criterio
rows_np = np.array(rows, dtype=object)

print("Top 10 por fora do intervalo:")
for r in sorted(rows, key=lambda x: x[1], reverse=True)[:10]:
    print(r)

print("\nTop 10 por curvatura:")
for r in sorted(rows, key=lambda x: x[2], reverse=True)[:10]:
    print(r)

print("\nTop 10 por oscilacao:")
for r in sorted(rows, key=lambda x: x[3], reverse=True)[:10]:
    print(r)

print("\nTop 10 por instabilidade (max salto):")
for r in sorted(rows, key=lambda x: x[4], reverse=True)[:10]:
    print(r)


In [ ]:
# Tabela comparativa GRU (TSDF_GRU vs TS_GRU)
import json
from pathlib import Path
import pandas as pd

paths = [
    Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results/final_rebuild_gru_c-3.0_3.0_radam_lr0.0002.json'),
    Path('/home/ferna/CPE727-2025-03/Seminarios/6 - RNN/seqKAN_results/final_rebuild_gru_H24_c-3.0_3.0.json'),
]
rows = []
for p in paths:
    if not p.exists():
        rows.append({'file': str(p), 'status': 'missing'})
        continue
    data = json.loads(p.read_text())
    tb = data.get('Test_best') or {}
    tf = data.get('Test_final') or {}
    rows.append({
        'file': p.name,
        'best_epoch': data.get('best_epoch') or data.get('best_epoch_test_micro'),
        'best_test_micro': data.get('best_test_micro'),
        'best_test_micro_se': tb.get('micro_se'),
        'final_test_micro': tf.get('micro_mse'),
        'final_test_micro_se': tf.get('micro_se'),
        'L1_best': data.get('L1_best'),
        'L1_final': data.get('L1_final'),
    })

df = pd.DataFrame(rows)
print(df)
